In [2]:

from __future__ import annotations

import glob
import hashlib
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


# ---------------- PATHS ----------------
ROOT_DIR       = Path(r"/home/tsultan1/IROS/Data")
SYNC_PARENT    = Path(r"cleaned/synchronized_proper_lite_union_v3")
LABEL_DIRNAME  = "label"  # FULL timeline folder
CSV_GLOB       = "*_icml_consensus_labels.csv"

DATASET_DIR    = ROOT_DIR / "_dataset_icml_v1"
DATASET_DIR.mkdir(exist_ok=True)

MANIFEST_OUT   = DATASET_DIR / "manifest_v3.csv"
WINDOWS_BASE   = DATASET_DIR / "windows_base_v3.csv"
SPLITS_OUT     = DATASET_DIR / "splits_v3.csv"
FOLD_SUMMARY   = DATASET_DIR / "splits_v3_fold_summary.csv"
META_JSON      = DATASET_DIR / "splits_v3_meta.json"


# ---------------- LOSO + VALIDATION ----------------
# Keep LOSO: test subject held-out. Validation uses K subjects (disjoint from train/test).
VAL_K = 5                 # recommended 4–6
VAL_PICK_MODE = "seeded"  # "seeded" | "round_robin"
VAL_SEED_OFFSET = 12345   # for reproducibility


# ---------------- WINDOWING ----------------
SLIDE_WIN_S    = 2.0
SLIDE_STRIDE_S = 0.25

# Optional: onset-anchor windows for analysis (not recommended for standard training)
INCLUDE_ONSET_ANCHOR = True
ANCHOR_PRE_S  = 2.0
ANCHOR_POST_S = 3.0

# Transition guard: dilate is_transition (or derived edges) by ±guard
TRANSITION_GUARD_S = 0.25

# Explicit REST-safe and ACTION-safe guards from edges (onset/offset)
REST_EDGE_GUARD_S   = 0.75   # keep REST windows away from action boundaries
ACTION_EDGE_GUARD_S = 0.25   # keep ACTION windows away from boundaries (usually smaller)

# Purity thresholds (fraction of samples in window that satisfy safe mask)
REST_MIN_SAFE_FRAC   = 0.95
ACTION_MIN_SAFE_FRAC = 0.80

# Task label consistency inside ACTION-safe window
TASK_CONSISTENCY_MIN = 0.70   # fraction of action-safe samples matching the voted task
VALID_TASK_SET       = {1, 2, 3, 4, 5}

# Optional caps (deterministic per file) to avoid crazy huge windows_base
MAX_REST_WINDOWS_PER_FILE   = None  # e.g., 200
MAX_ACTION_WINDOWS_PER_FILE = None  # e.g., 200
MAX_ANCHOR_WINDOWS_PER_FILE = None  # e.g., 20


# ---------------- IO / DIAGNOSTICS ----------------
WRITE_FILE_MD5 = False
WRITE_SPLITS_INCREMENTAL = True  # strongly recommended

# Required + optional columns
REQ_MIN  = ["Timestamp_seconds", "subject_id", "task", "trial", "label_action", "task_target"]
OPT_COLS = ["active_prob", "w_label", "is_transition"]


# =============================================================================
# Helpers
# =============================================================================
def _is_subdir_name(name: str) -> bool:
    return re.match(r"(?i)^sub-?\d+$", name) is not None

def _file_md5(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as fh:
        while True:
            b = fh.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def _stable_seed_from_str(s: str) -> int:
    h = hashlib.md5(s.encode("utf-8")).hexdigest()
    return int(h[:8], 16)

def _to_int_safe(x, default: int = 0) -> int:
    v = pd.to_numeric(x, errors="coerce")
    if pd.isna(v):
        return int(default)
    try:
        fv = float(v)
        if not np.isfinite(fv):
            return int(default)
        return int(fv)
    except Exception:
        return int(default)

def _median_fs(t: np.ndarray) -> float:
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if dt.size == 0:
        return np.nan
    return 1.0 / np.median(dt)

def _read_df_minimal(csv_path: Path) -> pd.DataFrame:
    """Reads required + optional cols if present; sorts/dedups by time."""
    head = pd.read_csv(csv_path, nrows=0, low_memory=False)
    cols = list(head.columns)
    usecols = [c for c in (REQ_MIN + OPT_COLS) if c in cols]

    df = pd.read_csv(csv_path, low_memory=False, usecols=usecols)
    missing = [c for c in REQ_MIN if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path.name}: missing required columns {missing}")

    df["Timestamp_seconds"] = pd.to_numeric(df["Timestamp_seconds"], errors="coerce")
    df = df.dropna(subset=["Timestamp_seconds"]).sort_values("Timestamp_seconds")
    df = df.drop_duplicates(subset=["Timestamp_seconds"], keep="first").reset_index(drop=True)
    return df

def _collect_label_files_from_label_folder() -> List[Path]:
    files: List[Path] = []
    for sub in sorted(p for p in ROOT_DIR.iterdir() if p.is_dir() and _is_subdir_name(p.name)):
        label_dir = sub / SYNC_PARENT / LABEL_DIRNAME
        if not label_dir.exists():
            print(f"[skip] missing label dir: {label_dir}")
            continue
        got = sorted(Path(x) for x in glob.glob(str((label_dir / CSV_GLOB).resolve())))
        if not got:
            print(f"[skip] no label CSVs in: {label_dir}")
            continue
        print(f"[use] {sub.name} → label/ ({len(got)} files)")
        files.extend(got)
    return files

def _dilate_bool_mask(x: np.ndarray, radius: int) -> np.ndarray:
    if radius <= 0:
        return x.astype(bool)
    xi = (x.astype(int) > 0).astype(int)
    kernel = np.ones(2 * radius + 1, dtype=int)
    y = np.convolve(xi, kernel, mode="same") > 0
    return y.astype(bool)

def _edges_from_mask(mask01: np.ndarray) -> np.ndarray:
    """Indices where mask changes (onset OR offset)."""
    x = (mask01.astype(int) > 0).astype(int)
    d = np.diff(x)
    return np.where(d != 0)[0] + 1

def _onsets_from_mask(mask01: np.ndarray) -> np.ndarray:
    """Indices where mask rises 0->1."""
    x = (mask01.astype(int) > 0).astype(int)
    d = np.diff(np.r_[0, x])
    return np.where(d == 1)[0]

def _transition_mask_from_df(df: pd.DataFrame, fs: float, label_action: np.ndarray) -> np.ndarray:
    """
    Prefer Phase-2 `is_transition` BUT dilate by ±GUARD.
    Else derive from label_action edges and dilate by ±GUARD.
    """
    n = len(df)
    guard_n = max(1, int(round(TRANSITION_GUARD_S * fs)))

    if "is_transition" in df.columns:
        it = pd.to_numeric(df["is_transition"], errors="coerce").fillna(0).astype(int).to_numpy()
        it = (it > 0).astype(bool)
        return _dilate_bool_mask(it, guard_n)

    edges = _edges_from_mask(label_action)
    m = np.zeros(n, dtype=bool)
    for b in edges:
        lo = max(0, b - guard_n)
        hi = min(n, b + guard_n + 1)
        m[lo:hi] = True
    return m

def _distance_to_nearest_true(mask_true: np.ndarray) -> np.ndarray:
    """1D distance transform to nearest True index (in samples)."""
    n = mask_true.size
    if n == 0:
        return np.array([], dtype=np.float32)
    dist = np.full(n, 1e9, dtype=np.float32)
    idx = np.where(mask_true)[0]
    if idx.size == 0:
        return dist  # all far
    dist[idx] = 0.0
    for i in range(1, n):
        dist[i] = min(dist[i], dist[i - 1] + 1.0)
    for i in range(n - 2, -1, -1):
        dist[i] = min(dist[i], dist[i + 1] + 1.0)
    return dist

def _nearest_onset_meta(t: np.ndarray, onsets: np.ndarray, center_idx: int) -> Tuple[float, int, float]:
    """(dist_to_onset_s_signed, onset_idx_nearest, onset_time_nearest_s)"""
    if onsets.size == 0 or not (0 <= center_idx < t.size):
        return (np.nan, -1, np.nan)
    j = int(np.argmin(np.abs(onsets - center_idx)))
    o = int(onsets[j])
    dist = float(t[center_idx] - t[o])
    return (dist, o, float(t[o]))

def _choose_val_subjects(subjects: List[int], test_subject: int, k: int) -> List[int]:
    others = sorted([s for s in subjects if s != test_subject])
    if len(others) == 0:
        return []
    k = int(min(k, len(others)))

    if VAL_PICK_MODE == "round_robin":
        start = (int(test_subject) + 7) % len(others)
        picked = [others[(start + i) % len(others)] for i in range(k)]
        return sorted(picked)

    # seeded sampling (recommended): deterministic per fold
    rng = np.random.RandomState(int(test_subject) + int(VAL_SEED_OFFSET))
    return sorted(rng.choice(others, size=k, replace=False).tolist())

def _cap_rows_deterministic(rows: List[Dict], max_n: Optional[int], seed_key: str) -> List[Dict]:
    if max_n is None:
        return rows
    max_n = int(max_n)
    if len(rows) <= max_n:
        return rows
    rng = np.random.RandomState(_stable_seed_from_str(seed_key))
    keep_idx = set(rng.choice(np.arange(len(rows)), size=max_n, replace=False).tolist())
    return [r for i, r in enumerate(rows) if i in keep_idx]


# =============================================================================
# Window generation (fold-independent, “clean ICML setup”)
#   - action head uses REST vs ACTION windows
#   - task head uses subset of ACTION windows with consistent task (T1–T5)
# =============================================================================
def _build_windows_base_for_file(file_path: Path, df: pd.DataFrame) -> Tuple[List[Dict], Dict]:
    """
    Returns:
      windows: list of dict rows (fold-independent)
      stats: per-file stats dict
    """
    n = len(df)
    if n < 2:
        return [], {"n": n}

    subj      = _to_int_safe(df["subject_id"].iloc[0], default=-1)
    task_code = _to_int_safe(df["task"].iloc[0], default=-1)
    trial_id  = _to_int_safe(df["trial"].iloc[0], default=-1)

    t = pd.to_numeric(df["Timestamp_seconds"], errors="coerce").to_numpy()
    fs = _median_fs(t)
    if not np.isfinite(fs) or fs <= 0:
        return [], {"n": n, "fs": float(fs)}

    la = pd.to_numeric(df["label_action"], errors="coerce").fillna(0).astype(int).to_numpy()
    tt = pd.to_numeric(df["task_target"],  errors="coerce").fillna(0).astype(int).to_numpy()

    # transition mask (+ dilation)
    trans_mask = _transition_mask_from_df(df, fs, la)

    # edges/onsets for guards + meta
    edges  = _edges_from_mask(la)
    onsets = _onsets_from_mask(la)

    edge_mask = np.zeros(n, dtype=bool)
    edge_mask[edges] = True
    dist_edge_n = _distance_to_nearest_true(edge_mask)

    rest_guard_n   = int(round(REST_EDGE_GUARD_S   * fs))
    action_guard_n = int(round(ACTION_EDGE_GUARD_S * fs))

    safe_core = ~trans_mask

    # Explicit safe masks (per-sample)
    rest_safe_mask   = safe_core & (la == 0) & (dist_edge_n >= rest_guard_n)
    action_safe_mask = safe_core & (la == 1) & (dist_edge_n >= action_guard_n)

    # Optional confidence cols
    w_label = pd.to_numeric(df["w_label"], errors="coerce").to_numpy() if "w_label" in df.columns else None
    a_prob  = pd.to_numeric(df["active_prob"], errors="coerce").to_numpy() if "active_prob" in df.columns else None

    win_n    = max(1, int(round(SLIDE_WIN_S    * fs)))
    stride_n = max(1, int(round(SLIDE_STRIDE_S * fs)))

    file_abs  = str(file_path.resolve())
    trial_key = f"{int(subj)}_{int(task_code)}_{int(trial_id)}"

    windows: List[Dict] = []
    rest_rows: List[Dict] = []
    action_rows: List[Dict] = []
    anchor_rows: List[Dict] = []

    # --- sliding windows ---
    for s in range(0, n - win_n + 1, stride_n):
        e = s + win_n
        center = (s + e) // 2

        # Purity fractions
        rest_frac   = float(np.mean(rest_safe_mask[s:e]))
        action_frac = float(np.mean(action_safe_mask[s:e]))
        trans_frac  = float(np.mean(trans_mask[s:e]))

        # REST window (clean)
        if rest_frac >= REST_MIN_SAFE_FRAC:
            dist_to_onset_s, onset_idx_nearest, onset_time_nearest_s = _nearest_onset_meta(t, onsets, center)

            row = dict(
                file=file_abs,
                subject_id=int(subj),
                task_code=int(task_code),
                trial_id=int(trial_id),
                trial_key=str(trial_key),

                type="sliding_rest_safe",
                start_idx=int(s),
                end_idx=int(e),
                start_time_s=float(t[s]),
                end_time_s=float(t[e - 1]),
                center_time_s=float(t[center]),

                # Labels (clean ICML)
                label_action=0,     # REST
                task_target=0,      # REST
                task_target_5way=-1,
                use_for_action=1,
                use_for_task=0,

                # diagnostics
                trans_frac=float(trans_frac),
                rest_safe_frac=float(rest_frac),
                action_safe_frac=float(action_frac),
                task_consistency=np.nan,
                dist_to_onset_s=float(dist_to_onset_s),
                onset_idx_nearest=int(onset_idx_nearest),
                onset_time_nearest_s=float(onset_time_nearest_s),
                w_label_mean=float(np.nanmean(w_label[s:e])) if w_label is not None else np.nan,
                active_prob_mean=float(np.nanmean(a_prob[s:e])) if a_prob is not None else np.nan,
            )
            rest_rows.append(row)
            continue

        # ACTION window (clean enough for action head)
        if action_frac >= ACTION_MIN_SAFE_FRAC:
            # vote task from action-safe samples inside the window
            m = action_safe_mask[s:e]
            tt_win = tt[s:e]
            cand = tt_win[m]
            cand = cand[(cand > 0)]  # ignore zeros
            if cand.size > 0:
                voted = int(np.bincount(cand.astype(int)).argmax())
            else:
                voted = int(task_code)

            # consistency
            if cand.size > 0:
                consistency = float(np.mean(cand == voted))
            else:
                consistency = 0.0

            use_for_task = int((voted in VALID_TASK_SET) and (consistency >= TASK_CONSISTENCY_MIN))
            task_5way = int(voted - 1) if voted in VALID_TASK_SET else -1

            dist_to_onset_s, onset_idx_nearest, onset_time_nearest_s = _nearest_onset_meta(t, onsets, center)

            row = dict(
                file=file_abs,
                subject_id=int(subj),
                task_code=int(task_code),
                trial_id=int(trial_id),
                trial_key=str(trial_key),

                type="sliding_action_safe",
                start_idx=int(s),
                end_idx=int(e),
                start_time_s=float(t[s]),
                end_time_s=float(t[e - 1]),
                center_time_s=float(t[center]),

                # Labels (clean ICML)
                label_action=1,           # ACTION
                task_target=int(voted),   # 1..5 ideally
                task_target_5way=int(task_5way),
                use_for_action=1,
                use_for_task=int(use_for_task),

                # diagnostics
                trans_frac=float(trans_frac),
                rest_safe_frac=float(rest_frac),
                action_safe_frac=float(action_frac),
                task_consistency=float(consistency),
                dist_to_onset_s=float(dist_to_onset_s),
                onset_idx_nearest=int(onset_idx_nearest),
                onset_time_nearest_s=float(onset_time_nearest_s),
                w_label_mean=float(np.nanmean(w_label[s:e])) if w_label is not None else np.nan,
                active_prob_mean=float(np.nanmean(a_prob[s:e])) if a_prob is not None else np.nan,
            )
            action_rows.append(row)

    # deterministic caps (optional)
    rest_rows   = _cap_rows_deterministic(rest_rows,   MAX_REST_WINDOWS_PER_FILE,   seed_key=file_abs + "::REST")
    action_rows = _cap_rows_deterministic(action_rows, MAX_ACTION_WINDOWS_PER_FILE, seed_key=file_abs + "::ACT")

    # --- onset-anchor windows (optional, for analysis) ---
    if INCLUDE_ONSET_ANCHOR and onsets.size > 0:
        pre_n  = max(0, int(round(ANCHOR_PRE_S  * fs)))
        post_n = max(1, int(round(ANCHOR_POST_S * fs)))

        for o in onsets:
            s = int(o - pre_n)
            e = int(o + post_n)
            if s < 0 or e > n or e <= s:
                continue

            # label by onset
            onset_task = int(tt[o]) if (0 <= o < tt.size and int(tt[o]) > 0) else int(task_code)
            task_5way  = int(onset_task - 1) if onset_task in VALID_TASK_SET else -1

            row = dict(
                file=file_abs,
                subject_id=int(subj),
                task_code=int(task_code),
                trial_id=int(trial_id),
                trial_key=str(trial_key),

                type="onset_anchor",
                start_idx=int(s),
                end_idx=int(e),
                start_time_s=float(t[s]),
                end_time_s=float(t[e - 1]),
                center_time_s=float(t[o]),

                label_action=1,
                task_target=int(onset_task),
                task_target_5way=int(task_5way),
                use_for_action=0,  # default off for standard training
                use_for_task=0,    # default off for standard training

                trans_frac=float(np.mean(trans_mask[s:e])),
                rest_safe_frac=float(np.mean(rest_safe_mask[s:e])),
                action_safe_frac=float(np.mean(action_safe_mask[s:e])),
                task_consistency=np.nan,
                dist_to_onset_s=0.0,
                onset_idx_nearest=int(o),
                onset_time_nearest_s=float(t[o]),
                w_label_mean=float(np.nanmean(w_label[s:e])) if w_label is not None else np.nan,
                active_prob_mean=float(np.nanmean(a_prob[s:e])) if a_prob is not None else np.nan,
            )
            anchor_rows.append(row)

        anchor_rows = _cap_rows_deterministic(anchor_rows, MAX_ANCHOR_WINDOWS_PER_FILE, seed_key=file_abs + "::ANC")

    windows = rest_rows + action_rows + anchor_rows

    stats = dict(
        file=file_abs,
        subject_id=int(subj),
        task_code=int(task_code),
        trial_id=int(trial_id),
        fs_hz=float(fs),
        n_samples=int(n),
        duration_s=float(t[-1] - t[0]) if t.size else 0.0,
        rest_frac=float(np.mean(la == 0)) if la.size else np.nan,
        action_frac=float(np.mean(la == 1)) if la.size else np.nan,
        n_edges=int(edges.size),
        n_onsets=int(onsets.size),
        n_windows_rest=int(len(rest_rows)),
        n_windows_action=int(len(action_rows)),
        n_windows_anchor=int(len(anchor_rows)),
    )
    return windows, stats


# =============================================================================
# Main: build manifest + windows_base + fold splits
# =============================================================================
def run_phase3_v3(
    manifest_out: Path = MANIFEST_OUT,
    windows_base_out: Path = WINDOWS_BASE,
    splits_out: Path = SPLITS_OUT,
    fold_summary_out: Path = FOLD_SUMMARY,
    meta_json_out: Path = META_JSON,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    files = _collect_label_files_from_label_folder()
    if not files:
        raise SystemExit("[stop] No label CSVs found in any subject's label/ folder.")

    # ---------- Build manifest + windows_base (once per file) ----------
    manifest_rows: List[Dict] = []
    windows_rows: List[Dict] = []
    bad: List[Tuple[str, str]] = []

    print("\n[phase3] Building manifest + base windows (fold-independent)...")
    for i, p in enumerate(files, 1):
        try:
            df = _read_df_minimal(p)
            # manifest info (file-level)
            t = df["Timestamp_seconds"].to_numpy()
            fs = _median_fs(t)
            dur = float(t[-1] - t[0]) if t.size else 0.0

            subj  = _to_int_safe(df["subject_id"].iloc[0], default=-1)
            task  = _to_int_safe(df["task"].iloc[0], default=-1)
            trial = _to_int_safe(df["trial"].iloc[0], default=-1)

            la = pd.to_numeric(df["label_action"], errors="coerce").fillna(0).astype(int).to_numpy()
            rest_frac = float(np.mean(la == 0)) if la.size else np.nan
            act_frac  = float(np.mean(la == 1)) if la.size else np.nan

            row = dict(
                file=str(p.resolve()),
                subject_id=int(subj),
                fs_hz=round(float(fs), 6) if np.isfinite(fs) else np.nan,
                duration_s=round(float(dur), 6),
                task_code=int(task),
                trial_id=int(trial),
                fold_id=int(subj),  # LOSO fold id = subject id
                rest_frac=round(float(rest_frac), 6) if np.isfinite(rest_frac) else np.nan,
                action_frac=round(float(act_frac), 6) if np.isfinite(act_frac) else np.nan,
            )
            if WRITE_FILE_MD5:
                row["md5"] = _file_md5(p)
            manifest_rows.append(row)

            # base windows
            w_rows, w_stats = _build_windows_base_for_file(p, df)
            windows_rows.extend(w_rows)

            if (i % 200) == 0:
                print(f"  ..processed {i}/{len(files)} files | windows_base so far: {len(windows_rows):,}")

        except Exception as e:
            bad.append((str(p), str(e)))

    manifest = pd.DataFrame(manifest_rows).sort_values(["subject_id", "file"]).reset_index(drop=True)
    manifest.to_csv(manifest_out, index=False)
    print(f"\n[ok] manifest → {manifest_out}  ({len(manifest)} ok, {len(bad)} skipped)")

    if bad:
        skipped = DATASET_DIR / "manifest_v3_skipped.csv"
        pd.DataFrame(bad, columns=["file", "reason"]).to_csv(skipped, index=False)
        print(f"[info] skipped manifest files → {skipped}")

    windows_base = pd.DataFrame(windows_rows)
    # stable columns
    base_cols = [
        "file","subject_id","task_code","trial_id","trial_key",
        "type","start_idx","end_idx","start_time_s","end_time_s","center_time_s",
        "label_action","task_target","task_target_5way","use_for_action","use_for_task",
        "trans_frac","rest_safe_frac","action_safe_frac","task_consistency",
        "dist_to_onset_s","onset_idx_nearest","onset_time_nearest_s",
        "w_label_mean","active_prob_mean",
    ]
    keep = [c for c in base_cols if c in windows_base.columns]
    extra = [c for c in windows_base.columns if c not in keep]
    windows_base = windows_base[keep + extra].reset_index(drop=True)
    windows_base.to_csv(windows_base_out, index=False)
    print(f"[ok] windows_base → {windows_base_out}  ({len(windows_base):,} windows)")

    # quick global sanity
    print("\n[sanity] windows_base label_action counts:")
    print(windows_base["label_action"].value_counts(dropna=False).sort_index())
    print("\n[sanity] windows_base task_target counts (top):")
    print(windows_base["task_target"].value_counts(dropna=False).sort_index().head(10))

    # ---------- TRUE LOSO splits (fold-dependent) ----------
    subjects = sorted(manifest["subject_id"].unique().tolist())
    print(f"\nFound {len(subjects)} subjects: {subjects}")
    meta = dict(
        version="v3",
        val_k=int(VAL_K),
        val_pick_mode=str(VAL_PICK_MODE),
        val_seed_offset=int(VAL_SEED_OFFSET),
        slide_win_s=float(SLIDE_WIN_S),
        slide_stride_s=float(SLIDE_STRIDE_S),
        transition_guard_s=float(TRANSITION_GUARD_S),
        rest_edge_guard_s=float(REST_EDGE_GUARD_S),
        action_edge_guard_s=float(ACTION_EDGE_GUARD_S),
        rest_min_safe_frac=float(REST_MIN_SAFE_FRAC),
        action_min_safe_frac=float(ACTION_MIN_SAFE_FRAC),
        task_consistency_min=float(TASK_CONSISTENCY_MIN),
        include_onset_anchor=bool(INCLUDE_ONSET_ANCHOR),
        outputs=dict(
            manifest=str(manifest_out),
            windows_base=str(windows_base_out),
            splits=str(splits_out),
            fold_summary=str(fold_summary_out),
        ),
        folds={}
    )

    # incremental writing (recommended)
    if WRITE_SPLITS_INCREMENTAL:
        if splits_out.exists():
            splits_out.unlink()
        first_write = True

    fold_summary_rows: List[Dict] = []

    print("\n[phase3] Writing splits (fold-dependent)...")
    for f_i, test_subject in enumerate(subjects, 1):
        val_subjects = _choose_val_subjects(subjects, test_subject, VAL_K)
        val_set = set(val_subjects)

        meta["folds"][str(int(test_subject))] = dict(
            test_subject=int(test_subject),
            val_subjects=[int(x) for x in val_subjects]
        )

        subj_arr = windows_base["subject_id"].to_numpy()
        split = np.where(
            subj_arr == int(test_subject),
            "test",
            np.where(np.isin(subj_arr, np.array(sorted(val_set), dtype=subj_arr.dtype)), "val", "train")
        )

        df_fold = windows_base.copy()
        df_fold.insert(0, "fold_id", int(test_subject))
        df_fold.insert(1, "split", split.astype(str))
        df_fold["val_subjects"] = ",".join(map(str, val_subjects))

        # fold summary
        def _cnt(mask):
            return int(np.sum(mask))

        is_train = (df_fold["split"].values == "train")
        is_val   = (df_fold["split"].values == "val")
        is_test  = (df_fold["split"].values == "test")

        use_act  = (df_fold["use_for_action"].values.astype(int) == 1)
        use_task = (df_fold["use_for_task"].values.astype(int) == 1)

        fold_summary_rows.append(dict(
            fold_id=int(test_subject),
            test_subject=int(test_subject),
            val_subjects=",".join(map(str, val_subjects)),
            n_train=int(is_train.sum()),
            n_val=int(is_val.sum()),
            n_test=int(is_test.sum()),
            n_train_action=_cnt(is_train & use_act),
            n_val_action=_cnt(is_val & use_act),
            n_test_action=_cnt(is_test & use_act),
            n_train_task=_cnt(is_train & use_task),
            n_val_task=_cnt(is_val & use_task),
            n_test_task=_cnt(is_test & use_task),
            # REST/ACTION in val/test for debugging
            val_rest=_cnt(is_val & (df_fold["label_action"].values == 0)),
            val_action=_cnt(is_val & (df_fold["label_action"].values == 1)),
            test_rest=_cnt(is_test & (df_fold["label_action"].values == 0)),
            test_action=_cnt(is_test & (df_fold["label_action"].values == 1)),
        ))

        # write
        if WRITE_SPLITS_INCREMENTAL:
            df_fold.to_csv(splits_out, index=False, mode="w" if first_write else "a", header=first_write)
            first_write = False
        else:
            # if you really want it in RAM (not recommended), you can append and concat later
            raise RuntimeError("Set WRITE_SPLITS_INCREMENTAL=True (recommended).")

        if (f_i % 5) == 0 or f_i == len(subjects):
            print(f"  ..wrote fold {f_i}/{len(subjects)} (test={test_subject}, val_k={len(val_subjects)})")

    fold_summary_df = pd.DataFrame(fold_summary_rows)
    fold_summary_df.to_csv(fold_summary_out, index=False)
    with open(meta_json_out, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"\n[ok] splits → {splits_out}")
    print(f"[ok] fold summary → {fold_summary_out}")
    print(f"[ok] meta json → {meta_json_out}")

    # also return windows_base for quick inspection
    return manifest, windows_base


# ---------------- Notebook usage ----------------
manifest, windows_base = run_phase3_v3()

print("\nQuick checks:")
print("  windows_base types:", windows_base["type"].value_counts().to_dict())
print("  windows_base label_action:", windows_base["label_action"].value_counts().to_dict())
print("  windows_base use_for_task:", windows_base["use_for_task"].value_counts().to_dict())

# After running, your Phase-5 exporter should:
#   - use splits_v3.csv (instead of splits_v1.csv)
#   - for ACTION head: train/eval on rows with use_for_action==1 and label_action in {0,1}
#   - for TASK head:   train/eval on rows with use_for_task==1 and task_target in {1..5}
#     (optionally condition task head on action at inference)


[use] Sub-1 → label/ (83 files)
[use] Sub-10 → label/ (83 files)
[use] Sub-11 → label/ (82 files)
[use] Sub-12 → label/ (83 files)
[use] Sub-14 → label/ (83 files)
[use] Sub-15 → label/ (83 files)
[use] Sub-16 → label/ (82 files)
[use] Sub-18 → label/ (83 files)
[use] Sub-19 → label/ (83 files)
[use] Sub-21 → label/ (82 files)
[use] Sub-22 → label/ (83 files)
[use] Sub-23 → label/ (83 files)
[use] Sub-24 → label/ (83 files)
[use] Sub-25 → label/ (83 files)
[use] Sub-26 → label/ (83 files)
[use] Sub-27 → label/ (83 files)
[use] Sub-28 → label/ (83 files)
[use] Sub-3 → label/ (83 files)
[use] Sub-30 → label/ (83 files)
[use] Sub-32 → label/ (83 files)
[use] Sub-33 → label/ (83 files)
[use] Sub-34 → label/ (83 files)
[use] Sub-35 → label/ (83 files)
[use] Sub-36 → label/ (84 files)
[use] Sub-4 → label/ (82 files)
[use] Sub-5 → label/ (83 files)
[use] Sub-6 → label/ (83 files)
[use] Sub-7 → label/ (83 files)
[use] Sub-8 → label/ (83 files)
[use] Sub-9 → label/ (83 files)

[phase3] Building